# Qwen GRPO CPU smoke check

Runs one posterior SFT step, one posterior GRPO step, and one capped evaluation move while keeping a single Qwen instance loaded. The Kaggle run has a 10-minute limit.


In [ ]:
import os, subprocess
repo = '/kaggle/working/minesweeper-grpo'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', 'https://github.com/Anchal-T/minesweeper-grpo.git', repo], check=True)
print(f'[SETUP] repository={repo}')


In [ ]:
import os, subprocess, sys, threading, time
os.chdir('/kaggle/working/minesweeper-grpo')
env = os.environ.copy()
env.update({'PY': sys.executable, 'CPU_SMOKE': '1'})
started = time.monotonic()
print('[SMOKE] launching posterior SFT -> posterior GRPO -> capped evaluation (single model load)', flush=True)
proc = subprocess.Popen(['bash', 'kaggle/run.sh'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def stream_output():
    for line in proc.stdout:
        print(line, end="", flush=True)
reader = threading.Thread(target=stream_output, daemon=True)
reader.start()
try:
    return_code = proc.wait(timeout=540)
except subprocess.TimeoutExpired:
    proc.kill()
    proc.wait()
    reader.join()
    raise
reader.join()
if return_code:
    raise subprocess.CalledProcessError(return_code, proc.args)
print(f'[SMOKE] complete elapsed_seconds={time.monotonic() - started:.1f}', flush=True)
